In [1]:
!nvidia-smi          # doit afficher Tesla T4, ~15 Go
import torch
print(torch.cuda.is_available())   # True

Fri Oct 16 14:27:04 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# Cellule 1 — installation (Colab n'a pas l'env uv)
!pip install -q transformers datasets evaluate sentence-transformers

# Cellule 2 — données + tokenisation (identique au local)
from datasets import load_dataset
from transformers import AutoTokenizer

ledgar = load_dataset("coastalcph/lex_glue", "ledgar")
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=256)

tok = ledgar.map(tokenize, batched=True)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.3 MB/s eta 0:00:00


README.md:   0%|          | 0.00/34.1k [00:00<?, ?B/s]

ledgar/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 20.9MB            

ledgar/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

ledgar/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 3.31MB            

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/60000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

In [3]:
# Cellule 3 — modèle + Trainer (identique au local, avec le GPU en plus)
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding
import evaluate, numpy as np

model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=100)
collator = DataCollatorWithPadding(tokenizer=tokenizer)
acc_m, f1_m = evaluate.load("accuracy"), evaluate.load("f1")

def compute_metrics(pred):
    preds = pred.predictions.argmax(-1)
    return {"accuracy": acc_m.compute(predictions=preds, references=pred.label_ids)["accuracy"],
            "f1_macro": f1_m.compute(predictions=preds, references=pred.label_ids, average="macro")["f1"]}

args = TrainingArguments(output_dir="runs/bert-ledgar",
                         learning_rate=2e-5, per_device_train_batch_size=16,
                         num_train_epochs=2, eval_strategy="epoch",
                         save_strategy="epoch", logging_steps=100)

trainer = Trainer(model=model, args=args,
                  train_dataset=tok["train"], eval_dataset=tok["validation"],
                  data_collator=collator, compute_metrics=compute_metrics)
trainer.train()

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [ ]:
# Cellule 4 — sauvegarde du modèle entraîné
trainer.save_model("bert-ledgar")
tokenizer.save_pretrained("bert-ledgar")
!zip -r bert-ledgar.zip bert-ledgar/
# puis : icône dossier à gauche → télécharger bert-ledgar.zip

In [ ]:
# Cellule 5 — Sauvegarde permanente sur Google Drive

from google.colab import drive
from pathlib import Path
import shutil

drive.mount("/content/drive")

source = Path("bert-ledgar")
destination = Path("/content/drive/MyDrive/Projet_Deep_Learning/bert-ledgar")

destination.parent.mkdir(parents=True, exist_ok=True)

shutil.copytree(source, destination, dirs_exist_ok=True)

print("Modèle sauvegardé sur Google Drive :", destination)

print("Fichiers enregistrés :")
for fichier in destination.iterdir():
    print("-", fichier.name)

In [ ]:
pred = trainer.predict(tokenized["test"])
print(pred.metrics)   # test_accuracy, test_f1_macro